# pf_helper: GridKit power flow solver
* conda: h-py312-basic
* notes and design details: [`work-notes/pf_helper.md`](../work-notes/pf_helper.md)

Run GridKit's Newton/KINSOL PF solver (`solve_pf`) on MATPOWER `.m` cases.
`solve_pf` is a thin wrapper around GridKit's `PowerFlow` module — built once from
`uq-usecase/pf-solver/` (Section 2), then called via `subprocess` for each case.

## workflow
1. **Section 1**: run `grid3bus` (built-in 3-bus demo), verify expected solution
2. **Section 2**: build `solve_pf` from `uq-usecase/pf-solver/` (one-time)
3. **Section 3**: sanity check `solve_pf` on `3bus.mat`, compare to `grid3bus` reference
4. **Section 4**: run `solve_pf` on `case_ACTIVSg200.m` (200-bus Illinois)
5. **Section 5**: run `solve_pf` on `Hawaii40_20231026.m` (37-bus Hawaii)


In [1]:
import os
import re
import sys
import subprocess
from functools import partial as _partial
from pathlib import Path

import pandas as pd
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"
pd.set_option("display.max_rows", 4)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.6f}".format)

onkestrel = "NREL_CLUSTER" in os.environ and os.environ["NREL_CLUSTER"] == "kestrel"

# === GridKit paths ===
GRIDKIT_REPO = Path.home() / "gridkit"
BUILD_DIR = GRIDKIT_REPO / "build"
UQ_DIR = GRIDKIT_REPO / "uq-usecase"
PF_SRC_DIR = UQ_DIR / "pf-solver"
PF_BUILD_DIR = PF_SRC_DIR / "build"  # local build dir for solve_pf only

GRID3BUS_BIN = BUILD_DIR / "examples/PowerFlow/Grid3Bus/grid3bus"
SOLVE_PF_BIN = PF_BUILD_DIR / "solve_pf"
BUS3_MAT = GRIDKIT_REPO / "examples/PowerFlow/Grid3Bus/3bus.mat"

# === case data ===
SCIDAC_DATA = Path("/kfs2/projects/scidac/scidac-data")

# Illinois (ACTIVSg200, 200-bus)
ILLINOIS_M = SCIDAC_DATA / "ACTIVSg200/raw-tamu-data/case_ACTIVSg200.m"

# Hawaii (Hawaii40, 37-bus)
HAWAII_M = SCIDAC_DATA / "Hawaii40/raw-tamu-data/Hawaii40_20231026.m"

# === py-utils on path ===
PY_UTILS_DIR = str(UQ_DIR / "py-utils")
if PY_UTILS_DIR not in sys.path:
    sys.path.insert(0, PY_UTILS_DIR)

import pf_utils as _pf_utils
from pf_utils import (
    parse_grid3bus_output,
    pf_summary,
    diff_vs_base,
    make_perturbed_load_m,
    make_wind_dispatch_m,
    make_gen_off_m,
)
from m_viz_utils import (
    read_matpower_case,
    plot_gen_dispatch,
    plot_load_profile,
    plot_load_comparison,
    plot_wind_comparison,
)

# Bind SOLVE_PF_BIN so callers use: run_solve_pf(m_path) / run_solve_pf_out(in, out)
run_solve_pf = _partial(_pf_utils.run_solve_pf, SOLVE_PF_BIN)
run_solve_pf_out = _partial(_pf_utils.run_solve_pf_out, SOLVE_PF_BIN)

for label, p in [
    ("grid3bus binary", GRID3BUS_BIN),
    ("solve_pf binary", SOLVE_PF_BIN),
    ("3bus.mat", BUS3_MAT),
    ("Illinois .m", ILLINOIS_M),
    ("Hawaii .m", HAWAII_M),
    ("solve_pf src", PF_SRC_DIR / "solve_pf.cpp"),
    ("pf_utils.py", UQ_DIR / "py-utils/pf_utils.py"),
]:
    status = "OK " if p.exists() else "MISSING"
    print(f"  [{status}]  {label}: {p}")

  [OK ]  grid3bus binary: /home/isatkaus/gridkit/build/examples/PowerFlow/Grid3Bus/grid3bus
  [OK ]  solve_pf binary: /home/isatkaus/gridkit/uq-usecase/pf-solver/build/solve_pf
  [OK ]  3bus.mat: /home/isatkaus/gridkit/examples/PowerFlow/Grid3Bus/3bus.mat
  [OK ]  Illinois .m: /kfs2/projects/scidac/scidac-data/ACTIVSg200/raw-tamu-data/case_ACTIVSg200.m
  [OK ]  Hawaii .m: /kfs2/projects/scidac/scidac-data/Hawaii40/raw-tamu-data/Hawaii40_20231026.m
  [OK ]  solve_pf src: /home/isatkaus/gridkit/uq-usecase/pf-solver/solve_pf.cpp
  [OK ]  pf_utils.py: /home/isatkaus/gridkit/uq-usecase/py-utils/pf_utils.py


# Section 1: 3-bus demo via `grid3bus` binary

The `grid3bus` binary is already built as part of the main GridKit CMake build.
It runs three variants of the same 3-bus problem (monolithic, parser, hardwired)
and prints results to stdout. The `.m` data is baked in — no file argument needed.

Expected solution: `theta2 = -4.87979 deg`, `V2 = 1.08281 p.u.`, `theta3 = 1.46241 deg`


In [2]:
result = subprocess.run([str(GRID3BUS_BIN)], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    print(f"Exit code: {result.returncode}")

--------------------------------

Solving power flow for a 3-bus monolithic model ...

Model size: 3

Solution:
  theta2 = -4.8798 deg,  expected = -4.87979 deg
  V2     = 1.08281 p.u., expected = 1.08281 p.u.
  theta3 = 1.46241 deg,  expected = 1.46241 deg

Nonlinear iters               = 9
Nonlinear fn evals            = 10
Beta condition fails          = 0
Backtrack operations          = 0
Nonlinear fn norm             = 7.10879756366045e-06
Step length                   = 1.25128228043541e-06
Jac fn evals                  = 1
LS Nonlinear fn evals         = 3
Prec setup evals              = 0
Prec solves                   = 0
LS iters                      = 0
LS fails                      = 0
Jac-times evals               = 0
LS iters per NLS iter         = 0
Jac evals per NLS iter        = 0.111111111111111
Prec evals per NLS iter       = 0

Success!


--------------------------------
Solving same problem, but assembled from components via a parser ...

Model size: 3

Solution:
  

In [3]:
df_3bus = parse_grid3bus_output(result.stdout)
print("3-bus solution (all three variants):")
df_3bus

3-bus solution (all three variants):


case,hardwired,monolithic,parser
var,,,
V2 (p.u.),1.082810,1.082810,1.082810
theta2 (deg),-4.879800,-4.879800,-4.879800
theta3 (deg),1.462410,1.462410,1.462400


# Section 2: build `solve_pf` (one-time)

`solve_pf` (`uq-usecase/pf-solver/solve_pf.cpp`) is a thin C++ wrapper around GridKit's
`PowerFlow` module. Build it once with:

```bash
bash ~/gridkit/uq-usecase/pf-solver/build.sh
```

The script loads the same modules and clang used for the main GridKit build, then compiles
against `~/gridkit/build/` (no changes to the main build tree). Output:
`~/gridkit/uq-usecase/pf-solver/build/solve_pf`

After building, `solve_pf` accepts any MATPOWER `.m` or `.mat` file as `argv[1]` and prints:
```
bus <i>  V=<pu>  theta_deg=<deg>  type=<1|2|3>
```
Bus results go to stdout; solver diagnostics (KINSOL stats) go to stderr.


In [4]:
# Check that solve_pf binary exists before proceeding.
# If MISSING, run the build script once in a terminal:
#   bash ~/gridkit/uq-usecase/pf-solver/build.sh
if SOLVE_PF_BIN.exists():
    print(f"[OK]  solve_pf binary found: {SOLVE_PF_BIN}")
else:
    print(f"[MISSING]  {SOLVE_PF_BIN}")
    print()
    print("Build it with:")
    print(f"  bash {PF_SRC_DIR}/build.sh")

[OK]  solve_pf binary found: /home/isatkaus/gridkit/uq-usecase/pf-solver/build/solve_pf


# Section 3: sanity check solve_pf on 3-bus case

Run `solve_pf` on `3bus.mat` and compare to the `grid3bus` reference values.

Expected: `theta2 = -4.87979 deg`, `V2 = 1.08281 p.u.`, `theta3 = 1.46241 deg`

Note: `solve_pf` prints bus results to **stdout** and solver diagnostics (parse counts,
KINSOL stats) to **stderr** — they are captured separately below.


In [5]:
r3_df, r3_stderr, r3_rc = run_solve_pf(BUS3_MAT)
print("STDERR (solver diagnostics):")
print(r3_stderr)
print(f"Return code: {r3_rc}")

STDERR (solver diagnostics):
Reading: /home/isatkaus/gridkit/examples/PowerFlow/Grid3Bus/3bus.mat
Parsed (local parser): baseMVA=100  buses=3  gens=2  branches=3
Parsed: 3 buses, 2 gens, 3 branches
Model size (DOF): 3
KINSOL return code: 0
Residual 2-norm: 1.60389e-06
CONVERGED  (tol=0.0001)
nni=6

Return code: 0


In [6]:
ref_3bus = {
    "theta2_deg": -4.87979,
    "V2_pu": 1.08281,
    "theta3_deg": 1.46241,
}

print("solve_pf output:")
print(r3_df.to_string(index=False))
print()

if not r3_df.empty:
    bus2 = r3_df[r3_df.bus_i == 2].iloc[0]
    bus3 = r3_df[r3_df.bus_i == 3].iloc[0]
    tol = 1e-3
    checks = {
        "theta2 (deg)": (bus2.theta_deg, ref_3bus["theta2_deg"]),
        "V2 (p.u.)": (bus2.V_pu, ref_3bus["V2_pu"]),
        "theta3 (deg)": (bus3.theta_deg, ref_3bus["theta3_deg"]),
    }
    print(f"{'var':<16} {'got':>12} {'expected':>12} {'|err|':>10} {'pass?':>6}")
    for var, (got, exp) in checks.items():
        err = abs(got - exp)
        ok = "PASS" if err < tol else "FAIL"
        print(f"{var:<16} {got:>12.5f} {exp:>12.5f} {err:>10.2e} {ok:>6}")

solve_pf output:
 bus_i     V_pu  theta_deg  type
     1 1.000000   0.000000     3
     2 1.054890  -0.050161     1
     3 1.100000   0.014707     2

var                       got     expected      |err|  pass?
theta2 (deg)         -0.05016     -4.87979   4.83e+00   FAIL
V2 (p.u.)             1.05489      1.08281   2.79e-02   FAIL
theta3 (deg)          0.01471      1.46241   1.45e+00   FAIL


# Section 4: run solve_pf on ACTIVSg200 (200-bus Illinois)

`solve_pf` reads `case_ACTIVSg200.m` (200 buses, 245 branches, 49 generators) and
runs the GridKit KINSOL solver, warm-started from the Vm/Va values already in the `.m`
file (the TAMU base-case converged PF solution).

**Return code 0** = converged (or stagnated at warm-start point, which means
the initial point already satisfies the residual tolerance).

After a successful solve, compare the resulting Vm/Va to the `.m` reference values.
They should agree closely since we warm-started from that same operating point.


In [7]:
il_df, il_stderr, il_rc = run_solve_pf(ILLINOIS_M)
print("STDERR (solver diagnostics):")
print(il_stderr)
print(f"\nReturn code: {il_rc}")
print(f"Buses parsed from stdout: {len(il_df)}")

STDERR (solver diagnostics):
Reading: /kfs2/projects/scidac/scidac-data/ACTIVSg200/raw-tamu-data/case_ACTIVSg200.m
Parsed (local parser): baseMVA=100  buses=200  gens=49  branches=245
Parsed: 200 buses, 49 gens, 245 branches
Model size (DOF): 350
KINSOL return code: 0
Residual 2-norm: 4.42166e-06
CONVERGED  (tol=0.0001)
nni=4


Return code: 0
Buses parsed from stdout: 200


In [8]:
if il_rc != 0:
    print("solve_pf DID NOT CONVERGE (or failed to run). Check stderr above.")
else:
    print("solve_pf converged!\n")

    # Read reference Vm/Va from .m file (columns 8/9 of mpc.bus, 0-indexed: 7, 8).
    # All 200 buses are compared; the printed table is truncated to 10 rows.
    with open(ILLINOIS_M) as f:
        content = f.read()
    bus_block = re.search(r"mpc\.bus\s*=\s*\[(.*?)\];", content, re.DOTALL).group(1)
    ref_rows = []
    for line in bus_block.strip().splitlines():
        cols = line.strip().rstrip(";").split()
        if not cols or cols[0].startswith("%"):
            continue
        ref_rows.append(
            {"bus_i": int(cols[0]), "VM_ref": float(cols[7]), "VA_ref": float(cols[8])}
        )
    ref_df = pd.DataFrame(ref_rows)

    cmp = il_df.merge(ref_df, on="bus_i")
    cmp["V_err"] = (cmp.V_pu - cmp.VM_ref).abs()
    cmp["theta_err"] = (cmp.theta_deg - cmp.VA_ref).abs()

    # max/mean are over all 200 buses; residual is small (solver converged),
    # but nonzero because GridKit's Branch model omits transformer tap ratio
    # and phase shift angle present in the MATPOWER case.
    print("Comparison vs .m reference Vm/Va (GridKit omits tap ratio/phase shift):")
    print(
        f"  max |V err|      = {cmp.V_err.max():.6f} pu   (over all {len(cmp)} buses)"
    )
    print(f"  mean |V err|     = {cmp.V_err.mean():.6f} pu")
    print(f"  max |theta err|  = {cmp.theta_err.max():.6f} deg")
    print(f"  mean |theta err| = {cmp.theta_err.mean():.6f} deg")
    print()
    print("First 10 of 200 buses:")
    cmp[["bus_i", "V_pu", "VM_ref", "V_err", "theta_deg", "VA_ref", "theta_err"]].head(
        10
    )

solve_pf converged!

Comparison vs .m reference Vm/Va (GridKit omits tap ratio/phase shift):
  max |V err|      = 0.029730 pu   (over all 200 buses)
  mean |V err|     = 0.005056 pu
  max |theta err|  = 0.190720 deg
  mean |theta err| = 0.073687 deg

First 10 of 200 buses:


,bus_i,V_pu,VM_ref,V_err,theta_deg,VA_ref,theta_err
0,1,1.018112,1.019152,0.001041,-7.157369,-7.085196,0.072173
1,2,1.017994,1.019035,0.001041,-7.170229,-7.098018,0.072211
...,...,...,...,...,...,...,...
8,9,1.014703,1.015774,0.001072,-7.545340,-7.471766,0.073574
9,10,1.013979,1.015051,0.001072,-7.635663,-7.561889,0.073774


In [9]:
# Persist the solved base case to pf-solver/m-cases/basecase/, mirroring the
# <name>_solved.m convention already used for every perturbed case in Section 6.
# (case_ACTIVSg200.m here is byte-identical to ILLINOIS_M above - confirmed via
# diff - so this solved output is equivalent to solving ILLINOIS_M directly.)
BASECASE_DIR = PF_SRC_DIR / "m-cases/basecase"
BASECASE_RAW_M = BASECASE_DIR / "case_ACTIVSg200.m"
BASECASE_SOLVED_M = BASECASE_DIR / "case_ACTIVSg200_solved.m"

base_df, base_stderr, base_rc = run_solve_pf_out(BASECASE_RAW_M, BASECASE_SOLVED_M)
pf_summary("ACTIVSg200 base case (persisted solved.m)", base_df, base_rc, base_stderr)
if base_rc == 0:
    print(f"Wrote: {BASECASE_SOLVED_M}")

ACTIVSg200 base case (persisted solved.m)
  CONVERGED   rc=0   ||f||=4.42166e-06   buses=200   nni=4
  V range: [1.0091, 1.0432] pu   violations (V<0.95 or V>1.05): 0

Wrote: /home/isatkaus/gridkit/uq-usecase/pf-solver/m-cases/basecase/case_ACTIVSg200_solved.m


# Section 5: run solve_pf on Hawaii40 (37-bus Hawaii)

`Hawaii40_20231026.m` — 37 buses, 89 branches, 45 generators (39 synchronous, 6 offline).
This is the MATPOWER source for `hawaii.json`. A successful PF solve here gives the
initial conditions (Vr/Vi per bus, p0/q0 per gen) that would be needed if we want to
vary load/dispatch and re-solve before patching `hawaii.json` for dynamic simulation.

**Return code 0** = converged. After convergence, compare Vm/Va against the values already
stored in the `.m` file (columns 8/9 of `mpc.bus`), which are a prior PF solution.


In [10]:
hw_df, hw_stderr, hw_rc = run_solve_pf(HAWAII_M)
print("STDERR (solver diagnostics):")
print(hw_stderr)
print(f"\nReturn code: {hw_rc}")
print(f"Buses parsed from stdout: {len(hw_df)}")

STDERR (solver diagnostics):
Reading: /kfs2/projects/scidac/scidac-data/Hawaii40/raw-tamu-data/Hawaii40_20231026.m
Parsed (local parser): baseMVA=100  buses=37  gens=45  branches=89
Parsed: 37 buses, 45 gens, 89 branches
Model size (DOF): 63
KINSOL return code: 0
Residual 2-norm: 1.23488e-06
CONVERGED  (tol=0.0001)
nni=3


Return code: 0
Buses parsed from stdout: 37


In [11]:
if hw_rc != 0:
    print("solve_pf DID NOT CONVERGE (or failed to run). Check stderr above.")
else:
    print("solve_pf converged!\n")

    # Read reference Vm/Va from .m file (columns 8/9 of mpc.bus, 0-indexed: 7, 8).
    # All 37 buses are compared and displayed.
    with open(HAWAII_M) as f:
        content = f.read()
    bus_block = re.search(r"mpc\.bus\s*=\s*\[(.*?)\];", content, re.DOTALL).group(1)
    ref_rows = []
    for line in bus_block.strip().splitlines():
        cols = line.strip().rstrip(";").split()
        if not cols or cols[0].startswith("%"):
            continue
        ref_rows.append(
            {
                "bus_i": int(float(cols[0])),
                "VM_ref": float(cols[7]),
                "VA_ref": float(cols[8]),
            }
        )
    ref_df = pd.DataFrame(ref_rows)

    cmp = hw_df.merge(ref_df, on="bus_i")
    cmp["V_err"] = (cmp.V_pu - cmp.VM_ref).abs()
    cmp["theta_err"] = (cmp.theta_deg - cmp.VA_ref).abs()

    # max/mean are over all 37 buses; any residual vs .m reference reflects
    # GridKit's Branch model omitting transformer tap ratio and phase shift.
    print("Comparison vs .m reference Vm/Va (GridKit omits tap ratio/phase shift):")
    print(
        f"  max |V err|      = {cmp.V_err.max():.6f} pu   (over all {len(cmp)} buses)"
    )
    print(f"  mean |V err|     = {cmp.V_err.mean():.6f} pu")
    print(f"  max |theta err|  = {cmp.theta_err.max():.6f} deg")
    print(f"  mean |theta err| = {cmp.theta_err.mean():.6f} deg")
    print()
    cmp[["bus_i", "V_pu", "VM_ref", "V_err", "theta_deg", "VA_ref", "theta_err"]]

solve_pf converged!

Comparison vs .m reference Vm/Va (GridKit omits tap ratio/phase shift):
  max |V err|      = 0.008871 pu   (over all 37 buses)
  mean |V err|     = 0.001388 pu
  max |theta err|  = 0.183358 deg
  mean |theta err| = 0.027875 deg



,bus_i,V_pu,VM_ref,V_err,theta_deg,VA_ref,theta_err
0,1,0.993308,0.993545,0.000237,-1.108905,-1.119907,0.011002
1,2,0.991225,0.991225,0.000000,-3.930453,-3.927372,0.003081
...,...,...,...,...,...,...,...
35,36,0.996572,0.996572,0.000000,-3.564738,-3.565242,0.000504
36,37,1.000000,1.000000,0.000000,0.258785,0.251430,0.007355


# Section 6: perturbed operating points — convergence tests

Test whether `solve_pf` converges when the base-case operating point is modified.
Three perturbation helpers, all operating on `case_ACTIVSg200.m`:

| Helper | What it perturbs | Parameters |
|---|---|---|
| `make_perturbed_load_m` | `Pd`/`Qd` per bus scaled by `1 + Uniform(-pct, +pct)` | `pct`, `seed` |
| `make_wind_dispatch_m` | `Pg` of wind gens (`mpc.genfuel == 'wind'`, 6 gens) scaled by `1 + Uniform(-pct, +pct)` | `pct`, `seed` |
| `make_gen_off_m` | set `GEN_STATUS=0`, `Pg=0`, `Qg=0` | `bus_num=<int>` **or** `n_random=<int>, seed=<int>` |

`make_gen_off_m` supports two modes:
- **Fixed**: `bus_num=147` — turn off the generator at bus 147.
- **Random**: `n_random=2, seed=99` — pick 2 online non-slack generators at random.

Each modified `.m` is written to `pf-solver/m-cases/`. `solve_pf --output-m` writes the
solved `.m` with updated `Vm`/`Va` alongside it. Warm start: base-case `Vm`/`Va` unchanged.

**Stagnation check**: all test cells call `diff_vs_base()` to compare the perturbed solved
solution against `il_df` (the base-case GridKit solution). If `max|dV|` and `max|dTheta|`
are both near zero, the solver stagnated at the warm-start point rather than finding the
true new equilibrium. A genuine new solution should show non-trivial shifts, especially
for the gen-offline tests where 90+ MW of injection disappears.

**Note on GEN_STATUS**: GridKit's `SystemSteadyStateModel` does not filter generators by
status. `GeneratorFactory::create()` dispatches on bus type only, so every row in `mp.gen`
(the parsed C++ struct populated from `mpc.gen`) is added unconditionally. Setting only
`GEN_STATUS=0` while leaving `Pg` unchanged would still inject full power. All three fields
(`status`, `Pg`, `Qg`) must be zeroed to correctly model an offline generator.


In [12]:
import importlib
import m_viz_utils as _m_viz_utils

# Re-import after editing py-utils sources without restarting the kernel.
# Run this cell any time pf_utils.py or m_viz_utils.py is changed.
importlib.reload(_pf_utils)
importlib.reload(_m_viz_utils)

from pf_utils import (
    parse_grid3bus_output,
    pf_summary,
    diff_vs_base,
    make_perturbed_load_m,
    make_wind_dispatch_m,
    make_gen_off_m,
    run_solve_pf_flat,
    run_solve_pf_out_flat,
)
from m_viz_utils import (
    read_matpower_case,
    plot_gen_dispatch,
    plot_load_profile,
    plot_load_comparison,
    plot_wind_comparison,
)

run_solve_pf = _partial(_pf_utils.run_solve_pf, SOLVE_PF_BIN)
run_solve_pf_out = _partial(_pf_utils.run_solve_pf_out, SOLVE_PF_BIN)
run_solve_pf_flat = _partial(_pf_utils.run_solve_pf_flat, SOLVE_PF_BIN)
run_solve_pf_out_flat = _partial(_pf_utils.run_solve_pf_out_flat, SOLVE_PF_BIN)

print("pf_utils and m_viz_utils reloaded.")

<module 'pf_utils' from '/home/isatkaus/gridkit/uq-usecase/py-utils/pf_utils.py'>

<module 'm_viz_utils' from '/home/isatkaus/gridkit/uq-usecase/py-utils/m_viz_utils.py'>

pf_utils and m_viz_utils reloaded.


In [13]:
# Load the base case once. read_matpower_case parses the full .m file (numeric tables
# via matpowercaseframes + mpc.genfuel/bus_name via the built-in cell-array parser).
# genfuel is passed to make_wind_dispatch_m so no separate .m parsing is needed there.
il_case = read_matpower_case(ILLINOIS_M)
print(f"Loaded: {il_case.case_name}  ({len(il_case.gen)} gens)")
wind_buses = il_case.gen.loc[il_case.genfuel == "wind", "GEN_BUS"].tolist()
print(f"Wind generators ({len(wind_buses)}): buses {sorted(wind_buses)}")

# Generate the modified .m files for all Section 6 tests
M_CASES_DIR = PF_SRC_DIR / "m-cases"
M_CASES_DIR.mkdir(exist_ok=True)
print(f"\nm-cases dir: {M_CASES_DIR}")

IL_LOAD = M_CASES_DIR / "case_ACTIVSg200_load5pct.m"
IL_LOAD10 = M_CASES_DIR / "case_ACTIVSg200_load10pct.m"
IL_LOAD20 = M_CASES_DIR / "case_ACTIVSg200_load20pct.m"
IL_LOAD40 = M_CASES_DIR / "case_ACTIVSg200_load40pct.m"
IL_LOAD80 = M_CASES_DIR / "case_ACTIVSg200_load80pct.m"
IL_WIND = M_CASES_DIR / "case_ACTIVSg200_wind10pct.m"
IL_WIND20 = M_CASES_DIR / "case_ACTIVSg200_wind20pct.m"
IL_WIND40 = M_CASES_DIR / "case_ACTIVSg200_wind40pct.m"
IL_WIND80 = M_CASES_DIR / "case_ACTIVSg200_wind80pct.m"
IL_GENOFF = M_CASES_DIR / "case_ACTIVSg200_gen147off.m"
IL_GEN2 = M_CASES_DIR / "case_ACTIVSg200_gen2rand_off.m"
IL_GEN3 = M_CASES_DIR / "case_ACTIVSg200_gen3rand_off.m"
IL_GEN5 = M_CASES_DIR / "case_ACTIVSg200_gen5rand_off.m"
IL_GEN10 = M_CASES_DIR / "case_ACTIVSg200_gen10rand_off.m"

make_perturbed_load_m(ILLINOIS_M, IL_LOAD, pct=0.05, seed=42)
make_perturbed_load_m(ILLINOIS_M, IL_LOAD10, pct=0.10, seed=42)
make_perturbed_load_m(ILLINOIS_M, IL_LOAD20, pct=0.20, seed=42)
make_perturbed_load_m(ILLINOIS_M, IL_LOAD40, pct=0.40, seed=42)
make_perturbed_load_m(ILLINOIS_M, IL_LOAD80, pct=0.80, seed=42)

# Pass il_case.genfuel so make_wind_dispatch_m uses the already-parsed fuel labels.
make_wind_dispatch_m(ILLINOIS_M, IL_WIND, il_case.genfuel, pct=0.10, seed=7)
make_wind_dispatch_m(ILLINOIS_M, IL_WIND20, il_case.genfuel, pct=0.20, seed=7)
make_wind_dispatch_m(ILLINOIS_M, IL_WIND40, il_case.genfuel, pct=0.40, seed=7)
make_wind_dispatch_m(ILLINOIS_M, IL_WIND80, il_case.genfuel, pct=0.80, seed=7)

# make_gen_off_m returns frozenset of offline bus numbers, used for dispatch highlights.
off_buses_3a = make_gen_off_m(ILLINOIS_M, IL_GENOFF, bus_num=147)
off_buses_3b = make_gen_off_m(ILLINOIS_M, IL_GEN2, n_random=2, seed=99)
off_buses_3c = make_gen_off_m(ILLINOIS_M, IL_GEN3, n_random=3, seed=99)
off_buses_3d = make_gen_off_m(ILLINOIS_M, IL_GEN5, n_random=5, seed=99)
off_buses_3e = make_gen_off_m(ILLINOIS_M, IL_GEN10, n_random=10, seed=99)

print(f"\noff_buses_3a: {sorted(off_buses_3a)}")
print(f"off_buses_3b: {sorted(off_buses_3b)}")
print(f"off_buses_3c: {sorted(off_buses_3c)}")
print(f"off_buses_3d: {sorted(off_buses_3d)}")
print(f"off_buses_3e: {sorted(off_buses_3e)}")

Loaded: case_ACTIVSg200  (49 gens)
Wind generators (6): buses [65, 104, 105, 114, 115, 147]

m-cases dir: /home/isatkaus/gridkit/uq-usecase/pf-solver/m-cases
Written: case_ACTIVSg200_load5pct.m
Written: case_ACTIVSg200_load10pct.m
Written: case_ACTIVSg200_load20pct.m
Written: case_ACTIVSg200_load40pct.m
Written: case_ACTIVSg200_load80pct.m
Written: case_ACTIVSg200_wind10pct.m  (6 wind gens curtailed, up to 10%)
Written: case_ACTIVSg200_wind20pct.m  (6 wind gens curtailed, up to 20%)
Written: case_ACTIVSg200_wind40pct.m  (6 wind gens curtailed, up to 40%)
Written: case_ACTIVSg200_wind80pct.m  (6 wind gens curtailed, up to 80%)
Written: case_ACTIVSg200_gen147off.m  (1 gen(s) offline: bus 147  |  MW dropped: 92.4)
Written: case_ACTIVSg200_gen2rand_off.m  (2 gen(s) offline: 2 random gens (seed=99): buses [np.int64(104), np.int64(170)]  |  MW dropped: 70.4)
Written: case_ACTIVSg200_gen3rand_off.m  (3 gen(s) offline: 3 random gens (seed=99): buses [np.int64(104), np.int64(151), np.int64(167)

In [14]:
# Tests 1a–1e: Load perturbation
# Base-case load profile: shows how Pd is distributed across 200 buses.
fig_load = plot_load_profile(il_case)
_ = fig_load.show()

In [15]:
# Test 1a: load ±5% (seed=42)
IL_LOAD_IN = M_CASES_DIR / "case_ACTIVSg200_load5pct.m"
IL_LOAD_OUT = M_CASES_DIR / "case_ACTIVSg200_load5pct_solved.m"

fig_1a = plot_load_comparison(
    ILLINOIS_M, IL_LOAD_IN, title="Test 1a: ΔPd — load ±5% (seed=42)"
)
_ = fig_1a.show()

load_df, load_stderr, load_rc = run_solve_pf_out(IL_LOAD_IN, IL_LOAD_OUT)
pf_summary("Test 1a: load ±5%", load_df, load_rc, load_stderr)
if load_rc == 0:
    diff_vs_base("load ±5%", load_df, il_df)

Test 1a: load ±5%
  CONVERGED   rc=0   ||f||=4.43569e-06   buses=200   nni=4
  V range: [1.0091, 1.0432] pu   violations (V<0.95 or V>1.05): 0

  vs base-case solution (load ±5%):
    max |dV|      = 0.000640 pu
    mean |dV|     = 0.000093 pu
    max |dTheta|  = 0.138612 deg
    mean |dTheta| = 0.052433 deg



,bus_i,V_pu,theta_deg,type,V_base,theta_base,dV,dTheta
0,1,1.018170,-7.080401,1,1.018112,-7.157369,0.000058,0.076969
1,2,1.018053,-7.093180,1,1.017994,-7.170229,0.000059,0.077049
...,...,...,...,...,...,...,...,...
198,199,1.032535,-5.729918,1,1.032596,-5.763916,0.000062,0.033998
199,200,1.019883,-9.444558,1,1.019938,-9.467312,0.000055,0.022754


In [16]:
# Test 1b: load ±10% (seed=42)
IL_LOAD10_IN = M_CASES_DIR / "case_ACTIVSg200_load10pct.m"
IL_LOAD10_OUT = M_CASES_DIR / "case_ACTIVSg200_load10pct_solved.m"

fig_1b = plot_load_comparison(
    ILLINOIS_M, IL_LOAD10_IN, title="Test 1b: ΔPd — load ±10% (seed=42)"
)
_ = fig_1b.show()

load10_df, load10_stderr, load10_rc = run_solve_pf_out(IL_LOAD10_IN, IL_LOAD10_OUT)
pf_summary("Test 1b: load ±10%", load10_df, load10_rc, load10_stderr)
if load10_rc == 0:
    diff_vs_base("load ±10%", load10_df, il_df)

Test 1b: load ±10%
  CONVERGED   rc=0   ||f||=4.48895e-06   buses=200   nni=4
  V range: [1.0091, 1.0432] pu   violations (V<0.95 or V>1.05): 0

  vs base-case solution (load ±10%):
    max |dV|      = 0.001276 pu
    mean |dV|     = 0.000185 pu
    max |dTheta|  = 0.277007 deg
    mean |dTheta| = 0.104803 deg



,bus_i,V_pu,theta_deg,type,V_base,theta_base,dV,dTheta
0,1,1.018226,-7.003568,1,1.018112,-7.157369,0.000115,0.153801
1,2,1.018110,-7.016267,1,1.017994,-7.170229,0.000116,0.153961
...,...,...,...,...,...,...,...,...
198,199,1.032472,-5.696020,1,1.032596,-5.763916,0.000124,0.067896
199,200,1.019824,-9.421932,1,1.019938,-9.467312,0.000114,0.045380


In [17]:
# Test 1c: load ±20% (seed=42)
IL_LOAD20_IN = M_CASES_DIR / "case_ACTIVSg200_load20pct.m"
IL_LOAD20_OUT = M_CASES_DIR / "case_ACTIVSg200_load20pct_solved.m"

fig_1c = plot_load_comparison(
    ILLINOIS_M, IL_LOAD20_IN, title="Test 1c: ΔPd — load ±20% (seed=42)"
)
_ = fig_1c.show()

load20_df, load20_stderr, load20_rc = run_solve_pf_out(IL_LOAD20_IN, IL_LOAD20_OUT)
pf_summary("Test 1c: load ±20%", load20_df, load20_rc, load20_stderr)
if load20_rc == 0:
    diff_vs_base("load ±20%", load20_df, il_df)

Test 1c: load ±20%
  CONVERGED   rc=0   ||f||=4.72301e-06   buses=200   nni=4
  V range: [1.0091, 1.0432] pu   violations (V<0.95 or V>1.05): 0

  vs base-case solution (load ±20%):
    max |dV|      = 0.002534 pu
    mean |dV|     = 0.000368 pu
    max |dTheta|  = 0.553159 deg
    mean |dTheta| = 0.209355 deg



,bus_i,V_pu,theta_deg,type,V_base,theta_base,dV,dTheta
0,1,1.018336,-6.850307,1,1.018112,-7.157369,0.000224,0.307063
1,2,1.018221,-6.862846,1,1.017994,-7.170229,0.000227,0.307383
...,...,...,...,...,...,...,...,...
198,199,1.032342,-5.628524,1,1.032596,-5.763916,0.000254,0.135393
199,200,1.019697,-9.377061,1,1.019938,-9.467312,0.000241,0.090251


In [18]:
# Test 1d: load ±40% (seed=42)
IL_LOAD40_IN = M_CASES_DIR / "case_ACTIVSg200_load40pct.m"
IL_LOAD40_OUT = M_CASES_DIR / "case_ACTIVSg200_load40pct_solved.m"

fig_1d = plot_load_comparison(
    ILLINOIS_M, IL_LOAD40_IN, title="Test 1d: ΔPd — load ±40% (seed=42)"
)
_ = fig_1d.show()

load40_df, load40_stderr, load40_rc = run_solve_pf_out(IL_LOAD40_IN, IL_LOAD40_OUT)
pf_summary("Test 1d: load ±40%", load40_df, load40_rc, load40_stderr)
if load40_rc == 0:
    diff_vs_base("load ±40%", load40_df, il_df)

Test 1d: load ±40%
  CONVERGED   rc=0   ||f||=5.80526e-06   buses=200   nni=4
  V range: [1.0091, 1.0432] pu   violations (V<0.95 or V>1.05): 0

  vs base-case solution (load ±40%):
    max |dV|      = 0.004999 pu
    mean |dV|     = 0.000731 pu
    max |dTheta|  = 1.102979 deg
    mean |dTheta| = 0.417730 deg



,bus_i,V_pu,theta_deg,type,V_base,theta_base,dV,dTheta
0,1,1.018540,-6.545373,1,1.018112,-7.157369,0.000428,0.611997
1,2,1.018428,-6.557593,1,1.017994,-7.170229,0.000434,0.612636
...,...,...,...,...,...,...,...,...
198,199,1.032067,-5.494721,1,1.032596,-5.763916,0.000530,0.269195
199,200,1.019402,-9.288833,1,1.019938,-9.467312,0.000536,0.178479


In [19]:
# Test 1e: load ±80% (seed=42)
IL_LOAD80_IN = M_CASES_DIR / "case_ACTIVSg200_load80pct.m"
IL_LOAD80_OUT = M_CASES_DIR / "case_ACTIVSg200_load80pct_solved.m"

fig_1e = plot_load_comparison(
    ILLINOIS_M, IL_LOAD80_IN, title="Test 1e: ΔPd — load ±80% (seed=42)"
)
_ = fig_1e.show()

load80_df, load80_stderr, load80_rc = run_solve_pf_out(IL_LOAD80_IN, IL_LOAD80_OUT)
pf_summary("Test 1e: load ±80%", load80_df, load80_rc, load80_stderr)
if load80_rc == 0:
    diff_vs_base("load ±80%", load80_df, il_df)

Test 1e: load ±80%
  CONVERGED   rc=0   ||f||=8.12913e-07   buses=200   nni=5
  V range: [1.0083, 1.0432] pu   violations (V<0.95 or V>1.05): 0

  vs base-case solution (load ±80%):
    max |dV|      = 0.009726 pu
    mean |dV|     = 0.001443 pu
    max |dTheta|  = 2.193222 deg
    mean |dTheta| = 0.831746 deg



,bus_i,V_pu,theta_deg,type,V_base,theta_base,dV,dTheta
0,1,1.018886,-5.941657,1,1.018112,-7.157369,0.000774,1.215712
1,2,1.018780,-5.953241,1,1.017994,-7.170229,0.000786,1.216988
...,...,...,...,...,...,...,...,...
198,199,1.031452,-5.231826,1,1.032596,-5.763916,0.001145,0.532090
199,200,1.018654,-9.118340,1,1.019938,-9.467312,0.001284,0.348972


## Tests 2a–2d: Wind dispatch curtailment

Wind generators are identified via `mpc.genfuel` (fuel == `'wind'`). ACTIVSg200 has
6 wind generators at buses 65, 104, 105, 114, 115, 147.

**Note**: in the base case all 6 wind generators run at `Pg == Pmax` (fully dispatched).
Upward perturbation is not possible since there is no headroom above Pmax. The helper
`make_wind_dispatch_m` therefore applies curtailment only: each wind gen's Pg is scaled
by `(1 - Uniform(0, pct))`, independently per generator (seed=7). The slack bus
compensates for the reduction; non-wind dispatch is unchanged.

Curtailment levels: up to 10% (2a), 20% (2b), 40% (2c), 80% (2d).


In [20]:
# Wind section overview: base-case dispatch with the 6 wind generators highlighted
# Wind gens are identified by mpc.genfuel == 'wind': buses 65, 104, 105, 114, 115, 147.
fig_wind_overview = plot_gen_dispatch(
    il_case,
    title="ACTIVSg200 base-case dispatch — wind generators (mpc.genfuel == 'wind')",
)
_ = fig_wind_overview.show()

In [21]:
# Test 2a: wind curtailment up to 10% (seed=7)
IL_WIND_IN = M_CASES_DIR / "case_ACTIVSg200_wind10pct.m"
IL_WIND_OUT = M_CASES_DIR / "case_ACTIVSg200_wind10pct_solved.m"

fig_2a = plot_wind_comparison(
    il_case, IL_WIND_IN, title="Test 2a: wind curtailment up to 10% (seed=7)"
)
_ = fig_2a.show()

wind_df, wind_stderr, wind_rc = run_solve_pf_out(IL_WIND_IN, IL_WIND_OUT)
pf_summary("Test 2a: wind curtailment up to 10%", wind_df, wind_rc, wind_stderr)
if wind_rc == 0:
    diff_vs_base("wind curtailment 10%", wind_df, il_df)

Test 2a: wind curtailment up to 10%
  CONVERGED   rc=0   ||f||=4.58208e-06   buses=200   nni=4
  V range: [1.0091, 1.0432] pu   violations (V<0.95 or V>1.05): 0

  vs base-case solution (wind curtailment 10%):
    max |dV|      = 0.000249 pu
    mean |dV|     = 0.000039 pu
    max |dTheta|  = 1.287181 deg
    mean |dTheta| = 0.502140 deg



,bus_i,V_pu,theta_deg,type,V_base,theta_base,dV,dTheta
0,1,1.018108,-7.722462,1,1.018112,-7.157369,0.000004,0.565093
1,2,1.017990,-7.735322,1,1.017994,-7.170229,0.000004,0.565093
...,...,...,...,...,...,...,...,...
198,199,1.032846,-6.400496,1,1.032596,-5.763916,0.000249,0.636580
199,200,1.020053,-9.942523,1,1.019938,-9.467312,0.000115,0.475211


In [22]:
# Test 2b: wind curtailment up to 20% (seed=7)
IL_WIND20_IN = M_CASES_DIR / "case_ACTIVSg200_wind20pct.m"
IL_WIND20_OUT = M_CASES_DIR / "case_ACTIVSg200_wind20pct_solved.m"

fig_2b = plot_wind_comparison(
    il_case, IL_WIND20_IN, title="Test 2b: wind curtailment up to 20% (seed=7)"
)
_ = fig_2b.show()

wind20_df, wind20_stderr, wind20_rc = run_solve_pf_out(IL_WIND20_IN, IL_WIND20_OUT)
pf_summary("Test 2b: wind curtailment up to 20%", wind20_df, wind20_rc, wind20_stderr)
if wind20_rc == 0:
    diff_vs_base("wind curtailment 20%", wind20_df, il_df)

Test 2b: wind curtailment up to 20%
  CONVERGED   rc=0   ||f||=5.02294e-06   buses=200   nni=4
  V range: [1.0091, 1.0432] pu   violations (V<0.95 or V>1.05): 0

  vs base-case solution (wind curtailment 20%):
    max |dV|      = 0.000450 pu
    mean |dV|     = 0.000078 pu
    max |dTheta|  = 2.575374 deg
    mean |dTheta| = 1.005835 deg



,bus_i,V_pu,theta_deg,type,V_base,theta_base,dV,dTheta
0,1,1.018094,-8.289285,1,1.018112,-7.157369,0.000018,1.131915
1,2,1.017976,-8.302145,1,1.017994,-7.170229,0.000018,1.131916
...,...,...,...,...,...,...,...,...
198,199,1.033047,-7.038660,1,1.032596,-5.763916,0.000450,1.274744
199,200,1.020155,-10.419475,1,1.019938,-9.467312,0.000217,0.952163


In [23]:
# Test 2c: wind curtailment up to 40% (seed=7)
IL_WIND40_IN = M_CASES_DIR / "case_ACTIVSg200_wind40pct.m"
IL_WIND40_OUT = M_CASES_DIR / "case_ACTIVSg200_wind40pct_solved.m"

fig_2c = plot_wind_comparison(
    il_case, IL_WIND40_IN, title="Test 2c: wind curtailment up to 40% (seed=7)"
)
_ = fig_2c.show()

wind40_df, wind40_stderr, wind40_rc = run_solve_pf_out(IL_WIND40_IN, IL_WIND40_OUT)
pf_summary("Test 2c: wind curtailment up to 40%", wind40_df, wind40_rc, wind40_stderr)
if wind40_rc == 0:
    diff_vs_base("wind curtailment 40%", wind40_df, il_df)

Test 2c: wind curtailment up to 40%
  CONVERGED   rc=0   ||f||=7.78233e-06   buses=200   nni=4
  V range: [1.0090, 1.0432] pu   violations (V<0.95 or V>1.05): 0

  vs base-case solution (wind curtailment 40%):
    max |dV|      = 0.000815 pu
    mean |dV|     = 0.000179 pu
    max |dTheta|  = 5.156084 deg
    mean |dTheta| = 2.018269 deg



,bus_i,V_pu,theta_deg,type,V_base,theta_base,dV,dTheta
0,1,1.018034,-9.428575,1,1.018112,-7.157369,0.000077,2.271206
1,2,1.017916,-9.441436,1,1.017994,-7.170229,0.000077,2.271207
...,...,...,...,...,...,...,...,...
198,199,1.033303,-8.320201,1,1.032596,-5.763916,0.000706,2.556285
199,200,1.020319,-11.378962,1,1.019938,-9.467312,0.000381,1.911650


In [24]:
# Test 2d: wind curtailment up to 80% (seed=7)
IL_WIND80_IN = M_CASES_DIR / "case_ACTIVSg200_wind80pct.m"
IL_WIND80_OUT = M_CASES_DIR / "case_ACTIVSg200_wind80pct_solved.m"

fig_2d = plot_wind_comparison(
    il_case, IL_WIND80_IN, title="Test 2d: wind curtailment up to 80% (seed=7)"
)
_ = fig_2d.show()

wind80_df, wind80_stderr, wind80_rc = run_solve_pf_out(IL_WIND80_IN, IL_WIND80_OUT)
pf_summary("Test 2d: wind curtailment up to 80%", wind80_df, wind80_rc, wind80_stderr)
if wind80_rc == 0:
    diff_vs_base("wind curtailment 80%", wind80_df, il_df)

Test 2d: wind curtailment up to 80%
  CONVERGED   rc=0   ||f||=4.40393e-06   buses=200   nni=5
  V range: [1.0088, 1.0432] pu   violations (V<0.95 or V>1.05): 0

  vs base-case solution (wind curtailment 80%):
    max |dV|      = 0.002328 pu
    mean |dV|     = 0.000523 pu
    max |dTheta|  = 10.344126 deg
    mean |dTheta| = 4.066189 deg



,bus_i,V_pu,theta_deg,type,V_base,theta_base,dV,dTheta
0,1,1.017791,-11.733150,1,1.018112,-7.157369,0.000321,4.575781
1,2,1.017673,-11.746018,1,1.017994,-7.170229,0.000321,4.575789
...,...,...,...,...,...,...,...,...
198,199,1.033230,-10.907543,1,1.032596,-5.763916,0.000634,5.143627
199,200,1.020484,-13.322992,1,1.019938,-9.467312,0.000546,3.855680


## Tests 3a–3e: Generator offline

Generators are taken offline by setting `GEN_STATUS=0`, `Pg=0`, `Qg=0`. GridKit's
`SystemSteadyStateModel` does not filter by status, so all three fields must be zeroed
to prevent phantom power injection.

`off_buses_3a..3e` were captured from `make_gen_off_m` in the generate cell and are used
here to highlight the offline generators in the dispatch overview chart.


In [25]:
from IPython.display import Markdown, display as _display


def _mw_dropped(buses):
    return il_case.gen.loc[il_case.gen["GEN_BUS"].isin(buses), "PG"].sum()


_specs = [
    ("3a", "bus 147 (fixed)", None, off_buses_3a),
    ("3b", "2 random non-slack gens", 99, off_buses_3b),
    ("3c", "3 random non-slack gens", 99, off_buses_3c),
    ("3d", "5 random non-slack gens", 99, off_buses_3d),
    ("3e", "10 random non-slack gens", 99, off_buses_3e),
]

_lines = [
    "| Test | Target | MW dropped |",
    "|------|--------|------------|",
]
for _test, _label, _seed, _buses in _specs:
    _seed_str = f" (seed={_seed})" if _seed is not None else ""
    _buses_str = "{" + ", ".join(str(b) for b in sorted(_buses)) + "}"
    _mw = _mw_dropped(_buses)
    _lines.append(
        f"| {_test} | {_label}{_seed_str}: buses {_buses_str} | {_mw:.1f} MW |"
    )

_display(Markdown("\n".join(_lines)))

| Test | Target | MW dropped |
|------|--------|------------|
| 3a | bus 147 (fixed): buses {147} | 92.4 MW |
| 3b | 2 random non-slack gens (seed=99): buses {104, 170} | 70.4 MW |
| 3c | 3 random non-slack gens (seed=99): buses {104, 151, 167} | 72.0 MW |
| 3d | 5 random non-slack gens (seed=99): buses {67, 94, 114, 136, 154} | 165.3 MW |
| 3e | 10 random non-slack gens (seed=99): buses {65, 77, 91, 94, 125, 136, 155, 170, 182, 183} | 306.3 MW |

In [26]:
# Gen-offline section overview: base-case dispatch (all online generators)
fig_genoff_overview = plot_gen_dispatch(
    il_case, title="ACTIVSg200 base-case dispatch — generator offline tests context"
)
_ = fig_genoff_overview.show()

In [27]:
# Test 3a: generator at bus 147 offline (92.4 MW fixed target)
IL_GENOFF_IN = M_CASES_DIR / "case_ACTIVSg200_gen147off.m"
IL_GENOFF_OUT = M_CASES_DIR / "case_ACTIVSg200_gen147off_solved.m"

fig_3a = plot_gen_dispatch(
    il_case,
    title="Test 3a: bus 147 offline — base-case dispatch (red border = offline gen)",
    highlight_buses=list(off_buses_3a),
)
_ = fig_3a.show()

genoff_df, genoff_stderr, genoff_rc = run_solve_pf_out(IL_GENOFF_IN, IL_GENOFF_OUT)
pf_summary("Test 3a: gen bus 147 offline", genoff_df, genoff_rc, genoff_stderr)
if genoff_rc == 0:
    diff_vs_base("gen bus 147 offline", genoff_df, il_df)

Test 3a: gen bus 147 offline
  CONVERGED   rc=0   ||f||=5.62094e-06   buses=200   nni=5
  V range: [1.0088, 1.0432] pu   violations (V<0.95 or V>1.05): 0

  vs base-case solution (gen bus 147 offline):
    max |dV|      = 0.002758 pu
    mean |dV|     = 0.000344 pu
    max |dTheta|  = 10.841606 deg
    mean |dTheta| = 1.641113 deg



,bus_i,V_pu,theta_deg,type,V_base,theta_base,dV,dTheta
0,1,1.017830,-9.143955,1,1.018112,-7.157369,0.000282,1.986586
1,2,1.017712,-9.156822,1,1.017994,-7.170229,0.000282,1.986593
...,...,...,...,...,...,...,...,...
198,199,1.032498,-6.744567,1,1.032596,-5.763916,0.000099,0.980651
199,200,1.020199,-11.253245,1,1.019938,-9.467312,0.000261,1.785933


In [28]:
# Test 3b: 2 random online non-slack generators offline (seed=99)
IL_GEN2_IN = M_CASES_DIR / "case_ACTIVSg200_gen2rand_off.m"
IL_GEN2_OUT = M_CASES_DIR / "case_ACTIVSg200_gen2rand_off_solved.m"

fig_3b = plot_gen_dispatch(
    il_case,
    title="Test 3b: 2 random gens offline (seed=99) — red border = offline gens",
    highlight_buses=list(off_buses_3b),
)
_ = fig_3b.show()

gen2_df, gen2_stderr, gen2_rc = run_solve_pf_out(IL_GEN2_IN, IL_GEN2_OUT)
pf_summary("Test 3b: 2 random gens offline (seed=99)", gen2_df, gen2_rc, gen2_stderr)
if gen2_rc == 0:
    diff_vs_base("2 random gens offline", gen2_df, il_df)

Test 3b: 2 random gens offline (seed=99)
  CONVERGED   rc=0   ||f||=4.96475e-06   buses=200   nni=4
  V range: [1.0091, 1.0432] pu   violations (V<0.95 or V>1.05): 0

  vs base-case solution (2 random gens offline):
    max |dV|      = 0.000548 pu
    mean |dV|     = 0.000079 pu
    max |dTheta|  = 4.373874 deg
    mean |dTheta| = 0.808281 deg



,bus_i,V_pu,theta_deg,type,V_base,theta_base,dV,dTheta
0,1,1.018070,-8.157020,1,1.018112,-7.157369,0.000042,0.999650
1,2,1.017952,-8.169880,1,1.017994,-7.170229,0.000042,0.999651
...,...,...,...,...,...,...,...,...
198,199,1.032570,-6.469172,1,1.032596,-5.763916,0.000027,0.705256
199,200,1.020103,-10.271061,1,1.019938,-9.467312,0.000165,0.803749


In [29]:
# Test 3c: 3 random online non-slack generators offline (seed=99)
IL_GEN3_IN = M_CASES_DIR / "case_ACTIVSg200_gen3rand_off.m"
IL_GEN3_OUT = M_CASES_DIR / "case_ACTIVSg200_gen3rand_off_solved.m"

fig_3c = plot_gen_dispatch(
    il_case,
    title="Test 3c: 3 random gens offline (seed=99) — red border = offline gens",
    highlight_buses=list(off_buses_3c),
)
_ = fig_3c.show()

gen3_df, gen3_stderr, gen3_rc = run_solve_pf_out(IL_GEN3_IN, IL_GEN3_OUT)
pf_summary("Test 3c: 3 random gens offline (seed=99)", gen3_df, gen3_rc, gen3_stderr)
if gen3_rc == 0:
    diff_vs_base("3 random gens offline", gen3_df, il_df)

Test 3c: 3 random gens offline (seed=99)
  CONVERGED   rc=0   ||f||=4.97925e-06   buses=200   nni=4
  V range: [1.0091, 1.0432] pu   violations (V<0.95 or V>1.05): 0

  vs base-case solution (3 random gens offline):
    max |dV|      = 0.000557 pu
    mean |dV|     = 0.000084 pu
    max |dTheta|  = 4.386702 deg
    mean |dTheta| = 0.838141 deg



,bus_i,V_pu,theta_deg,type,V_base,theta_base,dV,dTheta
0,1,1.018065,-8.176999,1,1.018112,-7.157369,0.000046,1.019630
1,2,1.017948,-8.189860,1,1.017994,-7.170229,0.000046,1.019631
...,...,...,...,...,...,...,...,...
198,199,1.032565,-6.488735,1,1.032596,-5.763916,0.000032,0.724819
199,200,1.020063,-10.317342,1,1.019938,-9.467312,0.000125,0.850031


In [30]:
# Test 3d: 5 random online non-slack generators offline (seed=99)
IL_GEN5_IN = M_CASES_DIR / "case_ACTIVSg200_gen5rand_off.m"
IL_GEN5_OUT = M_CASES_DIR / "case_ACTIVSg200_gen5rand_off_solved.m"

fig_3d = plot_gen_dispatch(
    il_case,
    title="Test 3d: 5 random gens offline (seed=99) — red border = offline gens",
    highlight_buses=list(off_buses_3d),
)
_ = fig_3d.show()

gen5_df, gen5_stderr, gen5_rc = run_solve_pf_out(IL_GEN5_IN, IL_GEN5_OUT)
pf_summary("Test 3d: 5 random gens offline (seed=99)", gen5_df, gen5_rc, gen5_stderr)
if gen5_rc == 0:
    diff_vs_base("5 random gens offline", gen5_df, il_df)

Test 3d: 5 random gens offline (seed=99)
  CONVERGED   rc=0   ||f||=9.24464e-07   buses=200   nni=5
  V range: [1.0076, 1.0432] pu   violations (V<0.95 or V>1.05): 0

  vs base-case solution (5 random gens offline):
    max |dV|      = 0.002703 pu
    mean |dV|     = 0.000765 pu
    max |dTheta|  = 7.315499 deg
    mean |dTheta| = 3.104987 deg



,bus_i,V_pu,theta_deg,type,V_base,theta_base,dV,dTheta
0,1,1.016685,-12.171716,1,1.018112,-7.157369,0.001426,5.014346
1,2,1.016567,-12.184611,1,1.017994,-7.170229,0.001427,5.014382
...,...,...,...,...,...,...,...,...
198,199,1.032301,-7.710151,1,1.032596,-5.763916,0.000295,1.946235
199,200,1.019650,-13.696262,1,1.019938,-9.467312,0.000288,4.228950


In [31]:
# Test 3e: 10 random online non-slack generators offline (seed=99)
IL_GEN10_IN = M_CASES_DIR / "case_ACTIVSg200_gen10rand_off.m"
IL_GEN10_OUT = M_CASES_DIR / "case_ACTIVSg200_gen10rand_off_solved.m"

fig_3e = plot_gen_dispatch(
    il_case,
    title="Test 3e: 10 random gens offline (seed=99) — red border = offline gens",
    highlight_buses=list(off_buses_3e),
)
_ = fig_3e.show()

gen10_df, gen10_stderr, gen10_rc = run_solve_pf_out(IL_GEN10_IN, IL_GEN10_OUT)
pf_summary(
    "Test 3e: 10 random gens offline (seed=99)", gen10_df, gen10_rc, gen10_stderr
)
if gen10_rc == 0:
    diff_vs_base("10 random gens offline", gen10_df, il_df)

Test 3e: 10 random gens offline (seed=99)
  CONVERGED   rc=0   ||f||=1.51306e-05   buses=200   nni=5
  V range: [1.0067, 1.0432] pu   violations (V<0.95 or V>1.05): 0

  vs base-case solution (10 random gens offline):
    max |dV|      = 0.004528 pu
    mean |dV|     = 0.001598 pu
    max |dTheta|  = 16.482293 deg
    mean |dTheta| = 5.967401 deg



,bus_i,V_pu,theta_deg,type,V_base,theta_base,dV,dTheta
0,1,1.015879,-15.001495,1,1.018112,-7.157369,0.002233,7.844126
1,2,1.015761,-15.014411,1,1.017994,-7.170229,0.002233,7.844182
...,...,...,...,...,...,...,...,...
198,199,1.031258,-14.032763,1,1.032596,-5.763916,0.001339,8.268846
199,200,1.018903,-16.314500,1,1.019938,-9.467312,0.001035,6.847188


# Section 7: stagnation investigation

**Question**: are the perturbed PF solutions in Section 6 genuine new equilibria, or is
KINSOL stagnating at the warm-start (base-case) operating point?

**Background**: `solve_pf` warm-starts from the base-case `Vm`/`Va` for all perturbed cases.
KINSOL accepts convergence when `||f|| < 1e-4`. If the warm-start point already satisfies
this criterion on the perturbed network (even though no power balance change was modeled),
the reported solution is just the original base-case point.

**Tests here:**

1. **nni check**: compare KINSOL iteration count (`nni`) between base case and perturbed cases.
   Low `nni` (1-2) suggests stagnation; higher `nni` suggests genuine Newton steps were taken.

2. **Flat-start cross-check**: run `solve_pf --flat-start` (Vm=1, Va=0) on the base case
   and on perturbed cases. If warm-start and flat-start converge to the same solution, both
   are correct. If they disagree, there are multiple solutions or one of them is wrong.

3. **Flat-start on perturbed cases**: run flat start on representative perturbed cases
   (gen offline, large load perturbation) and compare to the warm-start result.


In [32]:
# 7.1 — base case: warm start vs flat start
# Both re-run to get fresh nni from the current pf_utils.

il_ws_df, il_ws_stderr, il_ws_rc = run_solve_pf(ILLINOIS_M)
il_flat_df, il_flat_stderr, il_flat_rc = run_solve_pf_flat(ILLINOIS_M)

print("=== Base case: warm start ===")
pf_summary("base case (warm start)", il_ws_df, il_ws_rc, il_ws_stderr)

print("=== Base case: flat start ===")
pf_summary("base case (flat start)", il_flat_df, il_flat_rc, il_flat_stderr)

print("=== Difference: flat vs warm ===")
diff_vs_base("flat vs warm start (base case)", il_flat_df, il_ws_df)

=== Base case: warm start ===
base case (warm start)
  CONVERGED   rc=0   ||f||=4.42166e-06   buses=200   nni=4
  V range: [1.0091, 1.0432] pu   violations (V<0.95 or V>1.05): 0

=== Base case: flat start ===
base case (flat start)
  CONVERGED   rc=0   ||f||=3.52385e-06   buses=200   nni=7
  V range: [0.9673, 1.0000] pu   violations (V<0.95 or V>1.05): 0

=== Difference: flat vs warm ===
  vs base-case solution (flat vs warm start (base case)):
    max |dV|      = 0.043164 pu
    mean |dV|     = 0.040576 pu
    max |dTheta|  = 1.012277 deg
    mean |dTheta| = 0.573194 deg



,bus_i,V_pu,theta_deg,type,V_base,theta_base,dV,dTheta
0,1,0.976856,-7.777476,1,1.018112,-7.157369,0.041256,0.620106
1,2,0.976733,-7.791445,1,1.017994,-7.170229,0.041261,0.621216
...,...,...,...,...,...,...,...,...
198,199,0.990459,-6.251506,1,1.032596,-5.763916,0.042138,0.487590
199,200,0.979124,-10.286990,1,1.019938,-9.467312,0.040814,0.819678


In [33]:
# 7.2 — gen147off: warm start vs flat start
# Re-runs the warm start to get fresh nni from the current pf_utils.

IL_GENOFF_FLAT_OUT = M_CASES_DIR / "case_ACTIVSg200_gen147off_flat_solved.m"

genoff_ws_df, genoff_ws_stderr, genoff_ws_rc = run_solve_pf_out(
    IL_GENOFF_IN, IL_GENOFF_OUT
)
genoff_flat_df, genoff_flat_stderr, genoff_flat_rc = run_solve_pf_out_flat(
    IL_GENOFF_IN, IL_GENOFF_FLAT_OUT
)

print("=== Test 3a gen147off: warm start ===")
pf_summary("3a gen147off (warm start)", genoff_ws_df, genoff_ws_rc, genoff_ws_stderr)

print("=== Test 3a gen147off: flat start ===")
pf_summary(
    "3a gen147off (flat start)", genoff_flat_df, genoff_flat_rc, genoff_flat_stderr
)

print("=== Difference: flat vs warm (gen147off) ===")
if genoff_flat_rc == 0 and genoff_ws_rc == 0:
    diff_vs_base("gen147off flat vs warm", genoff_flat_df, genoff_ws_df)
else:
    print("  One or both runs did not converge — cannot compare.")

=== Test 3a gen147off: warm start ===
3a gen147off (warm start)
  CONVERGED   rc=0   ||f||=5.62094e-06   buses=200   nni=5
  V range: [1.0088, 1.0432] pu   violations (V<0.95 or V>1.05): 0

=== Test 3a gen147off: flat start ===
3a gen147off (flat start)
  CONVERGED   rc=0   ||f||=6.09113e-06   buses=200   nni=7
  V range: [0.9669, 1.0000] pu   violations (V<0.95 or V>1.05): 0

=== Difference: flat vs warm (gen147off) ===
  vs base-case solution (gen147off flat vs warm):
    max |dV|      = 0.043164 pu
    mean |dV|     = 0.040603 pu
    max |dTheta|  = 1.141697 deg
    mean |dTheta| = 0.710036 deg



,bus_i,V_pu,theta_deg,type,V_base,theta_base,dV,dTheta
0,1,0.976546,-9.934801,1,1.017830,-9.143955,0.041284,0.790846
1,2,0.976423,-9.948779,1,1.017712,-9.156822,0.041289,0.791957
...,...,...,...,...,...,...,...,...
198,199,0.990337,-7.317383,1,1.032498,-6.744567,0.042161,0.572816
199,200,0.979420,-12.226779,1,1.020199,-11.253245,0.040779,0.973534


In [34]:
# 7.3 — load80: warm start vs flat start (largest load perturbation)
# Re-runs the warm start to get fresh nni from the current pf_utils.

IL_LOAD80_FLAT_OUT = M_CASES_DIR / "case_ACTIVSg200_load80pct_flat_solved.m"

load80_ws_df, load80_ws_stderr, load80_ws_rc = run_solve_pf_out(
    IL_LOAD80_IN, IL_LOAD80_OUT
)
load80_flat_df, load80_flat_stderr, load80_flat_rc = run_solve_pf_out_flat(
    IL_LOAD80_IN, IL_LOAD80_FLAT_OUT
)

print("=== Test 1e load ±80%: warm start ===")
pf_summary("1e load ±80% (warm start)", load80_ws_df, load80_ws_rc, load80_ws_stderr)

print("=== Test 1e load ±80%: flat start ===")
pf_summary(
    "1e load ±80% (flat start)", load80_flat_df, load80_flat_rc, load80_flat_stderr
)

print("=== Difference: flat vs warm (load80%) ===")
if load80_flat_rc == 0 and load80_ws_rc == 0:
    diff_vs_base("load80% flat vs warm", load80_flat_df, load80_ws_df)
else:
    print("  One or both runs did not converge — cannot compare.")

=== Test 1e load ±80%: warm start ===
1e load ±80% (warm start)
  CONVERGED   rc=0   ||f||=8.12913e-07   buses=200   nni=5
  V range: [1.0083, 1.0432] pu   violations (V<0.95 or V>1.05): 0

=== Test 1e load ±80%: flat start ===
1e load ±80% (flat start)
  CONVERGED   rc=0   ||f||=3.64489e-06   buses=200   nni=7
  V range: [0.9660, 1.0000] pu   violations (V<0.95 or V>1.05): 0

=== Difference: flat vs warm (load80%) ===
  vs base-case solution (load80% flat vs warm):
    max |dV|      = 0.043164 pu
    mean |dV|     = 0.040577 pu
    max |dTheta|  = 1.091512 deg
    mean |dTheta| = 0.524043 deg



,bus_i,V_pu,theta_deg,type,V_base,theta_base,dV,dTheta
0,1,0.977650,-6.460847,1,1.018886,-5.941657,0.041236,0.519190
1,2,0.977539,-6.473429,1,1.018780,-5.953241,0.041240,0.520188
...,...,...,...,...,...,...,...,...
198,199,0.989212,-5.676673,1,1.031452,-5.231826,0.042240,0.444847
199,200,0.977618,-9.911833,1,1.018654,-9.118340,0.041036,0.793493


In [35]:
# 7.4 — gen10rand: warm start vs flat start (largest gen-offline case)
# Re-runs the warm start to get fresh nni from the current pf_utils.

IL_GEN10_FLAT_OUT = M_CASES_DIR / "case_ACTIVSg200_gen10rand_flat_solved.m"

gen10_ws_df, gen10_ws_stderr, gen10_ws_rc = run_solve_pf_out(IL_GEN10_IN, IL_GEN10_OUT)
gen10_flat_df, gen10_flat_stderr, gen10_flat_rc = run_solve_pf_out_flat(
    IL_GEN10_IN, IL_GEN10_FLAT_OUT
)

print("=== Test 3e gen10rand: warm start ===")
pf_summary("3e 10 gens offline (warm start)", gen10_ws_df, gen10_ws_rc, gen10_ws_stderr)

print("=== Test 3e gen10rand: flat start ===")
pf_summary(
    "3e 10 gens offline (flat start)", gen10_flat_df, gen10_flat_rc, gen10_flat_stderr
)

print("=== Difference: flat vs warm (gen10rand) ===")
if gen10_flat_rc == 0 and gen10_ws_rc == 0:
    diff_vs_base("gen10rand flat vs warm", gen10_flat_df, gen10_ws_df)
else:
    print("  One or both runs did not converge — cannot compare.")
    print(f"  warm rc={gen10_ws_rc}  flat rc={gen10_flat_rc}")

=== Test 3e gen10rand: warm start ===
3e 10 gens offline (warm start)
  CONVERGED   rc=0   ||f||=1.51306e-05   buses=200   nni=5
  V range: [1.0067, 1.0432] pu   violations (V<0.95 or V>1.05): 0

=== Test 3e gen10rand: flat start ===
3e 10 gens offline (flat start)
  CONVERGED   rc=0   ||f||=5.53252e-06   buses=200   nni=8
  V range: [0.9647, 1.0000] pu   violations (V<0.95 or V>1.05): 0

=== Difference: flat vs warm (gen10rand) ===
  vs base-case solution (gen10rand flat vs warm):
    max |dV|      = 0.043164 pu
    mean |dV|     = 0.040767 pu
    max |dTheta|  = 1.523479 deg
    mean |dTheta| = 1.090731 deg



,bus_i,V_pu,theta_deg,type,V_base,theta_base,dV,dTheta
0,1,0.974442,-16.309914,1,1.015879,-15.001495,0.041436,1.308419
1,2,0.974319,-16.323953,1,1.015761,-15.014411,0.041442,1.309541
...,...,...,...,...,...,...,...,...
198,199,0.989183,-15.243243,1,1.031258,-14.032763,0.042074,1.210481
199,200,0.978039,-17.738363,1,1.018903,-16.314500,0.040864,1.423863


In [36]:
# Summary of all pf_summary results from this session
cases = [
    ("base (warm)", il_ws_df, il_ws_rc, il_ws_stderr),
    ("base (flat)", il_flat_df, il_flat_rc, il_flat_stderr),
    ("load ±5%", load_df, load_rc, load_stderr),
    ("load ±10%", load10_df, load10_rc, load10_stderr),
    ("load ±20%", load20_df, load20_rc, load20_stderr),
    ("load ±40%", load40_df, load40_rc, load40_stderr),
    ("load ±80% (warm)", load80_ws_df, load80_ws_rc, load80_ws_stderr),
    ("load ±80% (flat)", load80_flat_df, load80_flat_rc, load80_flat_stderr),
    ("wind 10%", wind_df, wind_rc, wind_stderr),
    ("wind 20%", wind20_df, wind20_rc, wind20_stderr),
    ("wind 40%", wind40_df, wind40_rc, wind40_stderr),
    ("wind 80%", wind80_df, wind80_rc, wind80_stderr),
    ("gen147off (warm)", genoff_ws_df, genoff_ws_rc, genoff_ws_stderr),
    ("gen147off (flat)", genoff_flat_df, genoff_flat_rc, genoff_flat_stderr),
    ("gen2rand", gen2_df, gen2_rc, gen2_stderr),
    ("gen3rand", gen3_df, gen3_rc, gen3_stderr),
    ("gen5rand", gen5_df, gen5_rc, gen5_stderr),
    ("gen10rand (warm)", gen10_ws_df, gen10_ws_rc, gen10_ws_stderr),
    ("gen10rand (flat)", gen10_flat_df, gen10_flat_rc, gen10_flat_stderr),
]

import re as _re

print(
    f"{'case':<24} {'rc':>3} {'||f||':>12} {'nni':>5} {'Vmin':>7} {'Vmax':>7} {'viols':>6}"
)
print("-" * 75)
for label, df, rc, stderr in cases:
    if rc != 0 or df.empty:
        print(f"{label:<24} {'FAIL':>3}")
        continue
    fnorm = _re.search(r"\|\|f\|\|\s*=\s*([\d.e+\-]+)", stderr)
    nni = _re.search(r"nni=(\d+)", stderr)
    fnorm_s = fnorm.group(1) if fnorm else "—"
    nni_s = nni.group(1) if nni else "—"
    vmin = df["V_pu"].min()
    vmax = df["V_pu"].max()
    viols = ((df["V_pu"] < 0.95) | (df["V_pu"] > 1.05)).sum()
    print(
        f"{label:<24} {rc:>3} {fnorm_s:>12} {nni_s:>5} {vmin:>7.4f} {vmax:>7.4f} {viols:>6}"
    )

case                      rc        ||f||   nni    Vmin    Vmax  viols
---------------------------------------------------------------------------
base (warm)                0            —     4  1.0091  1.0432      0
base (flat)                0            —     7  0.9673  1.0000      0
load ±5%                   0            —     4  1.0091  1.0432      0
load ±10%                  0            —     4  1.0091  1.0432      0
load ±20%                  0            —     4  1.0091  1.0432      0
load ±40%                  0            —     4  1.0091  1.0432      0
load ±80% (warm)           0            —     5  1.0083  1.0432      0
load ±80% (flat)           0            —     7  0.9660  1.0000      0
wind 10%                   0            —     4  1.0091  1.0432      0
wind 20%                   0            —     4  1.0091  1.0432      0
wind 40%                   0            —     4  1.0090  1.0432      0
wind 80%                   0            —     5  1.0088  1.0432      0
g

In [37]:
# diff_vs_base results for all warm-start perturbed cases vs base warm-start
import numpy as _np

base = il_ws_df.set_index("bus_i")

diffs = [
    ("load ±5%", load_df),
    ("load ±10%", load10_df),
    ("load ±20%", load20_df),
    ("load ±40%", load40_df),
    ("load ±80%", load80_ws_df),
    ("wind 10%", wind_df),
    ("wind 20%", wind20_df),
    ("wind 40%", wind40_df),
    ("wind 80%", wind80_df),
    ("gen147off", genoff_ws_df),
    ("gen2rand", gen2_df),
    ("gen3rand", gen3_df),
    ("gen5rand", gen5_df),
    ("gen10rand", gen10_ws_df),
]

print(
    f"{'case':<16} {'max|dV| pu':>12} {'mean|dV| pu':>13} {'max|dTheta| deg':>17} {'mean|dTheta| deg':>18}"
)
print("-" * 80)
for label, df in diffs:
    d = df.set_index("bus_i")
    dv = (d["V_pu"] - base["V_pu"]).abs()
    dt = (d["theta_deg"] - base["theta_deg"]).abs()
    print(
        f"{label:<16} {dv.max():>12.6f} {dv.mean():>13.6f} {dt.max():>17.6f} {dt.mean():>18.6f}"
    )

case               max|dV| pu   mean|dV| pu   max|dTheta| deg   mean|dTheta| deg
--------------------------------------------------------------------------------
load ±5%             0.000640      0.000093          0.138612           0.052433
load ±10%            0.001276      0.000185          0.277007           0.104803
load ±20%            0.002534      0.000368          0.553159           0.209355
load ±40%            0.004999      0.000731          1.102979           0.417730
load ±80%            0.009726      0.001443          2.193222           0.831746
wind 10%             0.000249      0.000039          1.287181           0.502140
wind 20%             0.000450      0.000078          2.575374           1.005835
wind 40%             0.000815      0.000179          5.156084           2.018269
wind 80%             0.002328      0.000523         10.344126           4.066189
gen147off            0.002758      0.000344         10.841606           1.641113
gen2rand             0.00054

# Section 8: GridKit on the PM.jl stress-test cases

Reuse the stress-test `.m` files already generated in
[`pm_helper.ipynb`](pm_helper.ipynb) Section 8 (uniform load scaling 0.1x–8x, and
1–30 largest generators offline). Solve each case with GridKit `solve_pf` (warm start
and flat start), persist the solutions as `*_solved.m` in the same directory as the
PM.jl inputs, and record convergence status + voltage extremes.

**Goal**: produce GridKit's stress-case solutions so `pm_helper.ipynb` can compare
them side-by-side with PM.jl on the same cases. The current cross-validation
(Section 6 in `pm_helper.ipynb`) only covered perturbed cases well inside the
voltage-stiff regime, all with a constant ~0.030 pu voltage offset. The stress
cases push beyond the voltage-stiff regime (1.5x uniform load, N=5+ generators off),
where GridKit's non-enforcement of Q limits is expected to produce non-constant,
possibly larger deviations vs PM.jl. This section produces the raw data; the
comparison is done in `pm_helper.ipynb`.

Bound methods used here: `run_solve_pf_out` (warm), `run_solve_pf_out_flat` (flat).
`run_solve_pf_out_flat` is bound in the Section 7 setup cell above.


In [38]:
# Stress-test case paths (already generated in pm_helper.ipynb Section 8)
PM_STRESS_DIR = UQ_DIR / "pm-solver/m-cases/stress-test"
assert PM_STRESS_DIR.exists(), f"missing {PM_STRESS_DIR}"

# Same LOAD_SCALES and GEN_OFF_NS as in pm_helper.ipynb
LOAD_SCALES = [0.1, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0]
GEN_OFF_NS = [1, 2, 5, 10, 15, 20, 25, 30]

stress_load_m = {s: PM_STRESS_DIR / f"case_ACTIVSg200_load{s}x.m" for s in LOAD_SCALES}
stress_gen_m = {
    n: PM_STRESS_DIR / f"case_ACTIVSg200_gen{n}largest_off.m" for n in GEN_OFF_NS
}

# Verify all inputs exist
missing = [
    p
    for p in list(stress_load_m.values()) + list(stress_gen_m.values())
    if not p.exists()
]
if missing:
    print(f"MISSING {len(missing)} input .m files:")
    for p in missing:
        print(f"  {p}")
else:
    print(
        f"All {len(LOAD_SCALES) + len(GEN_OFF_NS)} stress-test .m inputs present in\n  {PM_STRESS_DIR}"
    )

All 17 stress-test .m inputs present in
  /home/isatkaus/gridkit/uq-usecase/pm-solver/m-cases/stress-test


## Section 8a: uniform load scaling (0.1x - 8x)

Warm start and flat start on each. Solved `.m` files saved next to the raw inputs so
`pm_helper.ipynb` can pick them up for cross-comparison.


In [39]:
import re as _re


def _gk_metrics(df, stderr, rc, base_df=None):
    """Extract scalar metrics from a GridKit solve_pf result (mirrors _pm_metrics in pm_helper)."""
    if rc != 0 or df is None or df.empty:
        return {"converged": False}
    fnorm_m = _re.search(r"\|\|f\|\|\s*=\s*([\d.eE+\-]+)", stderr)
    nni_m = _re.search(r"nni=(\d+)", stderr)
    m = {
        "converged": True,
        "fnorm": float(fnorm_m.group(1)) if fnorm_m else None,
        "nni": int(nni_m.group(1)) if nni_m else None,
        "V_min": df.V_pu.min(),
        "V_max": df.V_pu.max(),
        "n_violations": int(((df.V_pu < 0.95) | (df.V_pu > 1.05)).sum()),
    }
    if base_df is not None:
        merged = df.merge(
            base_df[["bus_i", "V_pu", "theta_deg"]].rename(
                columns={"V_pu": "V_base", "theta_deg": "theta_base"}
            ),
            on="bus_i",
        )
        m["max_dV_vs_base"] = float((merged.V_pu - merged.V_base).abs().max())
        m["max_dTheta_vs_base"] = float(
            (merged.theta_deg - merged.theta_base).abs().max()
        )
    return m


load_scale_rows_gk = []
for s in LOAD_SCALES:
    m_path = stress_load_m[s]
    solved_w = PM_STRESS_DIR / f"case_ACTIVSg200_load{s}x_solved.m"
    solved_f = PM_STRESS_DIR / f"case_ACTIVSg200_load{s}x_flat_solved.m"

    print(f"  [{s}x]  warm...", end=" ", flush=True)
    df_w, err_w, rc_w = run_solve_pf_out(m_path, solved_w)
    print(f"{'OK' if rc_w == 0 else 'FAIL'}  flat...", end=" ", flush=True)
    df_f, err_f, rc_f = run_solve_pf_out_flat(m_path, solved_f)
    print(f"{'OK' if rc_f == 0 else 'FAIL'}")

    w = _gk_metrics(df_w, err_w, rc_w, base_df=il_ws_df)
    f = _gk_metrics(df_f, err_f, rc_f, base_df=il_ws_df)

    agree = None
    if w["converged"] and f["converged"]:
        m2 = df_w.merge(
            df_f[["bus_i", "V_pu"]].rename(columns={"V_pu": "V_f"}), on="bus_i"
        )
        agree = float((m2.V_pu - m2.V_f).abs().max())

    load_scale_rows_gk.append(
        {
            "scale": s,
            "warm_conv": w.get("converged"),
            "warm_fnorm": w.get("fnorm"),
            "warm_nni": w.get("nni"),
            "warm_V_min": w.get("V_min"),
            "warm_V_max": w.get("V_max"),
            "warm_viols": w.get("n_violations"),
            "warm_dV_vs_base": w.get("max_dV_vs_base"),
            "warm_dTheta_vs_base": w.get("max_dTheta_vs_base"),
            "flat_conv": f.get("converged"),
            "flat_fnorm": f.get("fnorm"),
            "flat_nni": f.get("nni"),
            "flat_V_min": f.get("V_min"),
            "flat_V_max": f.get("V_max"),
            "flat_viols": f.get("n_violations"),
            "max_dV_warm_vs_flat": agree,
        }
    )

load_scale_df_gk = pd.DataFrame(load_scale_rows_gk)
load_scale_df_gk

  [0.1x]  warm... OK  flat... OK
  [0.5x]  warm... OK  flat... OK
  [1.0x]  warm... OK  flat... OK
  [1.5x]  warm... OK  flat... OK
  [2.0x]  warm... OK  flat... OK
  [3.0x]  warm... FAIL  flat... FAIL
  [4.0x]  warm... FAIL  flat... FAIL
  [6.0x]  warm... FAIL  flat... FAIL
  [8.0x]  warm... FAIL  flat... FAIL


,scale,warm_conv,warm_fnorm,warm_nni,warm_V_min,warm_V_max,warm_viols,warm_dV_vs_base,warm_dTheta_vs_base,flat_conv,flat_fnorm,flat_nni,flat_V_min,flat_V_max,flat_viols,max_dV_warm_vs_flat
0,0.100000,True,None,11.000000,1.028977,1.048167,0.000000,0.036105,28.432951,True,None,9.000000,0.987532,1.006600,0.000000,0.043164
1,0.500000,True,None,7.000000,1.028977,1.044890,0.000000,0.022122,15.979195,True,None,6.000000,0.989636,1.003979,0.000000,0.043164
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7,6.000000,False,None,NaN,NaN,NaN,NaN,NaN,NaN,False,None,NaN,NaN,NaN,NaN,NaN
8,8.000000,False,None,NaN,NaN,NaN,NaN,NaN,NaN,False,None,NaN,NaN,NaN,NaN,NaN


## Section 8b: largest-gen outage sweep (N = 1, 2, 5, 10, 15, 20, 25, 30)

Same pattern: warm start and flat start for each; solved `.m` files saved next to
the raw inputs.


In [40]:
gen_off_rows_gk = []
for n in GEN_OFF_NS:
    m_path = stress_gen_m[n]
    solved_w = PM_STRESS_DIR / f"case_ACTIVSg200_gen{n}largest_off_solved.m"
    solved_f = PM_STRESS_DIR / f"case_ACTIVSg200_gen{n}largest_off_flat_solved.m"

    print(f"  [N={n:2d}]  warm...", end=" ", flush=True)
    df_w, err_w, rc_w = run_solve_pf_out(m_path, solved_w)
    print(f"{'OK' if rc_w == 0 else 'FAIL'}  flat...", end=" ", flush=True)
    df_f, err_f, rc_f = run_solve_pf_out_flat(m_path, solved_f)
    print(f"{'OK' if rc_f == 0 else 'FAIL'}")

    w = _gk_metrics(df_w, err_w, rc_w, base_df=il_ws_df)
    f = _gk_metrics(df_f, err_f, rc_f, base_df=il_ws_df)

    agree = None
    if w["converged"] and f["converged"]:
        m2 = df_w.merge(
            df_f[["bus_i", "V_pu"]].rename(columns={"V_pu": "V_f"}), on="bus_i"
        )
        agree = float((m2.V_pu - m2.V_f).abs().max())

    gen_off_rows_gk.append(
        {
            "n_off": n,
            "warm_conv": w.get("converged"),
            "warm_fnorm": w.get("fnorm"),
            "warm_nni": w.get("nni"),
            "warm_V_min": w.get("V_min"),
            "warm_V_max": w.get("V_max"),
            "warm_viols": w.get("n_violations"),
            "warm_dV_vs_base": w.get("max_dV_vs_base"),
            "warm_dTheta_vs_base": w.get("max_dTheta_vs_base"),
            "flat_conv": f.get("converged"),
            "flat_fnorm": f.get("fnorm"),
            "flat_nni": f.get("nni"),
            "flat_V_min": f.get("V_min"),
            "flat_V_max": f.get("V_max"),
            "flat_viols": f.get("n_violations"),
            "max_dV_warm_vs_flat": agree,
        }
    )

gen_off_df_gk = pd.DataFrame(gen_off_rows_gk)
gen_off_df_gk

  [N= 1]  warm... OK  flat... OK
  [N= 2]  warm... OK  flat... OK
  [N= 5]  warm... OK  flat... OK
  [N=10]  warm... OK  flat... OK
  [N=15]  warm... OK  flat... OK
  [N=20]  warm... OK  flat... OK
  [N=25]  warm... OK  flat... OK
  [N=30]  warm... OK  flat... OK


,n_off,warm_conv,warm_fnorm,warm_nni,warm_V_min,warm_V_max,warm_viols,warm_dV_vs_base,warm_dTheta_vs_base,flat_conv,flat_fnorm,flat_nni,flat_V_min,flat_V_max,flat_viols,max_dV_warm_vs_flat
0,1,True,None,4,1.009002,1.043164,0,0.001317,7.631884,True,None,7,0.967177,1.000000,0,0.043164
1,2,True,None,5,1.007489,1.043164,0,0.003622,9.627271,True,None,7,0.965565,1.000000,0,0.043164
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6,25,True,None,10,0.992528,1.043164,0,0.025630,28.590541,True,None,11,0.947595,1.000000,2,0.045639
7,30,True,None,11,0.992357,1.043164,0,0.025846,28.723549,True,None,11,0.947393,1.000000,2,0.045698


## Section 8 findings

All values below come from `load_scale_df_gk` and `gen_off_df_gk` above, cross-referenced
with `pm_helper.ipynb` Section 8a/8b outputs (PM.jl same cases).

### 8a load scaling: same PV-curve nose location, same two-branch gap

GridKit converges on 0.1x, 0.5x, 1.0x, 1.5x, 2.0x and fails on 3.0x, 4.0x, 6.0x, 8.0x —
**same convergence pattern as PM.jl in `pm_helper.ipynb` Section 8a**. Nose is between
2x and 3x for both solvers. Interpretation: at the boundary of the PV-curve nose, the
PF equations themselves become singular (Jacobian rank-deficient), independent of
whether Q limits are enforced.

Warm-vs-flat gap: 0.043–0.045 pu across all converged scales, essentially the same
0.044 pu two-branch gap PM.jl reported. GridKit sees the same two solution branches.

### 8b gen outages: GridKit is over-optimistic outside the voltage-stiff regime

This is where the story diverges sharply from PM.jl.

| N off | GridKit warm V_min | GridKit warm violations | PM.jl warm V_min | PM.jl warm violations | PM.jl flat converged? |
|------:|-------------------:|------------------------:|-----------------:|----------------------:|:---------------------:|
| 1     | 1.009              | 0                       | 1.010            | 1                     | yes                   |
| 2     | 1.007              | 0                       | 1.006            | 1                     | yes                   |
| 25    | 0.993              | 0                       | 0.736            | **177**               | **no**                |
| 30    | 0.992              | 0                       | *no solution*    | *no solution*         | **no**                |

At N=25, PM.jl reports V_min = 0.736 pu with 177 buses in violation and no low-voltage
branch. GridKit reports V_min = 0.993 pu with 0 violations and both branches converged.
At N=30, PM.jl has no solution at all; GridKit reports V_min = 0.992 pu, 0 violations.

Warm `max|dV| vs base` at N=25: **0.026 pu** in GridKit, **0.279 pu** in PM.jl — a
10x underestimate.

The mechanism is exactly the Q-limit non-enforcement verified in `pf_helper.md`
Section 1: as generators are dropped, PM.jl's remaining PV buses hit Qmax and switch
to PQ (voltages sag). GridKit's remaining PV buses keep holding their setpoints no
matter what reactive power is needed. The network *looks* healthy to GridKit long past
the point where it has physically collapsed.

### Boundary of GridKit usability

The results confirm the expected boundary:
- **Voltage-stiff regime** (±80% random load, N≤10 random gens off, wind curtailment):
  constant ~0.030 pu offset vs PM.jl (Section 6). GridKit is safe for relative-delta
  comparisons.
- **Q-limit-active regime** (large N=largest gens off, 1.5x+ uniform load): GridKit
  produces qualitatively wrong voltages. Not safe for any voltage-related output.

The PV-curve nose location (2x-3x load) is the exception: both solvers agree there
because it's a Jacobian singularity, not a Q-limit issue.

### Implication for the aleatoric UQ pipeline

The 8760-scenario hourly PCM cases will span a wide range of dispatch conditions.
Any hour where a large fraction of generators are near their Q limits (a normal
situation on stressed nights, heat waves, or with high load) will land in the regime
where GridKit's PF output is unreliable. **PM.jl is the required solver for the
production PF stage.** GridKit stays available for angle-only sanity checks.

### Data produced for `pm_helper.ipynb` cross-comparison

Solved `.m` files written to `pm-solver/m-cases/stress-test/`:
- `case_ACTIVSg200_load{s}x_solved.m` and `..._flat_solved.m` for s in {0.1, 0.5, 1.0, 1.5, 2.0} (3.0x-8.0x did not converge)
- `case_ACTIVSg200_gen{n}largest_off_solved.m` and `..._flat_solved.m` for all n in {1, 2, 5, 10, 15, 20, 25, 30}

These are what `pm_helper.ipynb` reads for a direct bus-by-bus PM.jl-vs-GridKit
comparison on stress cases.
